<a href="https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/shivam25th/flyrank-1st.git"
REPO = Path("/content/flyrank-1st")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )

os.chdir(REPO)

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

# Target is used only for evaluation, not for creating the score.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Repository:", REPO)
print("Dataset shape:", df.shape)



Repository: /content/flyrank-1st
Dataset shape: (30000, 45)


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def make_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("stale_page")

    if row["impressions_90d"] >= 500:
        reasons.append("visible_page")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.50
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if not reasons:
        reasons.append("general_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(make_reason_codes, axis=1)

print(df["reason_codes"].value_counts().head(10))

reason_codes
general_review                                         13083
visible_page|low_ctr_visible_page                       9731
visible_page                                            6931
stale_page                                               156
thin_visible_page                                         34
visible_page|thin_visible_page                            29
visible_page|low_ctr_visible_page|thin_visible_page       18
stale_page|visible_page|low_ctr_visible_page              10
stale_page|visible_page                                    7
stale_page|thin_visible_page                               1
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)
### Ranked review queue

I rank every page using the baseline refresh score. Higher scores indicate higher priority for human review under this rule.

The resulting queue is an ordered recommendation, not an automatic content-change decision.

In [ ]:
# 1. Visibility
df["visibility_score"] = (
    df["impressions_90d"].fillna(0).rank(pct=True)
)

# 2. Freshness risk
df["freshness_risk_score"] = (
    df["days_since_last_update"].fillna(0).rank(pct=True)
)

# 3. Position opportunity
position = df["avg_position"].fillna(df["avg_position"].median())

position_normalized = (
    (position - position.min()) /
    (position.max() - position.min())
)

df["position_opportunity_score"] = (
    (1 - position_normalized) *
    df["visibility_score"]
)

# 4. Content-depth gap
df["depth_gap_score"] = (
    (1 - df["word_count"].fillna(df["word_count"].median()).rank(pct=True))
    * df["visibility_score"]
)

# Final baseline score
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
)

print("Baseline score created successfully.")
print(df["baseline_refresh_score"].describe())

Baseline score created successfully.
count    30000.000000
mean         0.478304
std          0.224780
min          0.011741
25%          0.302479
50%          0.481434
75%          0.660946
max          0.968526
Name: baseline_refresh_score, dtype: float64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def make_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("stale_page")

    if row["impressions_90d"] >= 500:
        reasons.append("visible_page")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.50
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if not reasons:
        reasons.append("general_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(make_reason_codes, axis=1)

print(df["reason_codes"].value_counts().head(10))

reason_codes
general_review                                         13083
visible_page|low_ctr_visible_page                       9731
visible_page                                            6931
stale_page                                               156
thin_visible_page                                         34
visible_page|thin_visible_page                            29
visible_page|low_ctr_visible_page|thin_visible_page       18
stale_page|visible_page|low_ctr_visible_page              10
stale_page|visible_page                                    7
stale_page|thin_visible_page                               1
Name: count, dtype: int64


## 4. Weak picks + leakage check



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

queue_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

queue = (
    df[queue_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

output_dir = REPO / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows in queue:", len(queue))

print("\nTop 10 pages:")
print(queue.head(10).to_string(index=False))

Wrote: /content/flyrank-1st/work/outputs/baseline_action_score.csv
Rows in queue: 30000

Top 10 pages:
          content_id         client_id  baseline_rank  baseline_refresh_score  visibility_score  freshness_risk_score  position_opportunity_score  depth_gap_score                      reason_codes  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count
content_a5dbb404bdc2 client_f369cb89fc              1                0.968526          0.991300              0.991100                    0.956099         0.713026 visible_page|low_ctr_visible_page            79035           8.7 0.07               106                     106      2691.0
content_7368877ea310 client_7f2253d7e2              2                0.952500          0.986200              0.996033                    0.886372         0.752339           stale_page|visible_page            59472          24.8 0.13               231                     194      2591.0
content_6ac3ab740bbf client_f369cb89

In [ ]:
print(queue.shape)
print(queue.head(5))
print(output_path)

(30000, 15)
             content_id          client_id  baseline_rank  \
0  content_a5dbb404bdc2  client_f369cb89fc              1   
1  content_7368877ea310  client_7f2253d7e2              2   
2  content_6ac3ab740bbf  client_f369cb89fc              3   
3  content_cb7e312f5d32  client_9f14025af0              4   
4  content_fac19fcdfb85  client_4e07408562              5   

   baseline_refresh_score  visibility_score  freshness_risk_score  \
0                0.968526          0.991300              0.991100   
1                0.952500          0.986200              0.996033   
2                0.945346          0.948617              0.991100   
3                0.945302          0.944500              0.993567   
4                0.939054          0.996533              0.843200   

   position_opportunity_score  depth_gap_score  \
0                    0.956099         0.713026   
1                    0.886372         0.752339   
2                    0.930806         0.717360   
3     

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.